# Dunnhumby M1·M2·M3 모델 용량 진단
교수님의 언더피팅 가설을 검정하기 위해 동일한 candidate-specific N/V M2·M3 수식을 유지하고 ID 표현 차원만 64·128·256으로 늘립니다. 각 차원에서 M1·M2·M3를 처음부터 같은 조건으로 학습합니다. 신규상품 개발평가만 사용하며 final test·holdout은 만들지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REVIEWED_SHA = '0bfef95654b904a44808a02a25ac6b129fc112aa'
REPO = Path('/content/clv-m2-lightgcn-runner')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', REVIEWED_SHA], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
%cd /content/clv-m2-lightgcn-runner

In [ ]:
import json
import torch
import lightgcn_clv_candidate_nv_capacity_screen as screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert screen.CODE_VERSION == 'candidate-nv-m1-m2-m3-capacity-development-screen-v1'
cfg = screen.configure_capacity_screen()
summary = screen.preflight_summary(cfg)
assert cfg.capacity_dimensions == (64, 128, 256)
assert cfg.negative_count == 1 and cfg.epochs == 100 and cfg.n_layers == 2
assert len(summary['trained_models']) == 9 and summary['reused_models'] == []
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = screen.run_capacity_screen(cfg)

In [ ]:
from IPython.display import display
import pandas as pd

def show(value):
    frame = value.copy() if isinstance(value, pd.DataFrame) else pd.DataFrame(value)
    frame.attrs = {}
    display(frame)

core = [
    'model_id', 'axis', 'id_dim', 'trainable_parameter_count',
    'loss', 'p_correct', 'train_margin_mean', 'train_margin_positive_share',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'top10_share@10'
]
print('1) M1·M2·M3 용량별 전체 성능')
show(result_df[core])
print('2) 동일 용량 M1 대비 전체 지표')
show(result_df.attrs['comparison'])
print('3) 학습곡선')
show(result_df.attrs['training_history'])
print('4) 학습 양성-음성 점수 마진')
show(result_df.attrs['training_margin'])
print('5) Top-10 변경률')
show(result_df.attrs['top10_overlap'])
print('6) 용량부족 가설 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('7) 작동 진단')
print(json.dumps(result_df.attrs['mechanism_diagnostics'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))